In [1]:
# Movie Rating Prediction using Random Forest Regressor

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# --------------------------------------------------
# Load Dataset
# --------------------------------------------------

df = pd.read_csv("movie data.csv", encoding='latin1')

print("Dataset Shape:", df.shape)
print(df.head())

# --------------------------------------------------
# Data Cleaning
# --------------------------------------------------

# Remove rows where Rating is missing
df = df.dropna(subset=['Rating'])

# Clean Year column
df['Year'] = df['Year'].astype(str).str.extract('(\d{4})')
df['Year'] = pd.to_numeric(df['Year'], errors='coerce')

# Convert Duration to numeric
df['Duration'] = df['Duration'].astype(str).str.extract('(\d+)')
df['Duration'] = pd.to_numeric(df['Duration'], errors='coerce')

# Convert Votes to numeric
df['Votes'] = (
    df['Votes']
    .astype(str)
    .str.replace(',', '', regex=False)
)
df['Votes'] = pd.to_numeric(df['Votes'], errors='coerce')

# Fill missing values
categorical_cols = ['Genre', 'Director', 'Actor 1', 'Actor 2', 'Actor 3']

for col in categorical_cols:
    df[col] = df[col].fillna('Unknown')

numeric_cols = ['Year', 'Duration', 'Votes']

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# --------------------------------------------------
# Features and Target
# --------------------------------------------------

X = df[['Genre', 'Director', 'Actor 1', 'Actor 2',
        'Actor 3', 'Year', 'Duration', 'Votes']]

y = df['Rating']

# --------------------------------------------------
# Preprocessing
# --------------------------------------------------

categorical_features = [
    'Genre', 'Director',
    'Actor 1', 'Actor 2', 'Actor 3'
]

numerical_features = [
    'Year', 'Duration', 'Votes'
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            'cat',
            OneHotEncoder(handle_unknown='ignore'),
            categorical_features
        ),
        (
            'num',
            SimpleImputer(strategy='median'),
            numerical_features
        )
    ]
)

# --------------------------------------------------
# Model
# --------------------------------------------------

model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ))
])

# --------------------------------------------------
# Train-Test Split
# --------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# --------------------------------------------------
# Train Model
# --------------------------------------------------

model.fit(X_train, y_train)

# --------------------------------------------------
# Prediction
# --------------------------------------------------

y_pred = model.predict(X_test)

# --------------------------------------------------
# Evaluation
# --------------------------------------------------

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("\nModel Performance")
print("-" * 30)
print("MAE :", round(mae, 3))
print("MSE :", round(mse, 3))
print("RMSE:", round(rmse, 3))
print("R² Score:", round(r2, 3))

# --------------------------------------------------
# Example Movie Prediction
# --------------------------------------------------

new_movie = pd.DataFrame({
    'Genre': ['Action'],
    'Director': ['S. S. Rajamouli'],
    'Actor 1': ['Prabhas'],
    'Actor 2': ['Rana Daggubati'],
    'Actor 3': ['Anushka Shetty'],
    'Year': [2015],
    'Duration': [159],
    'Votes': [150000]
})

predicted_rating = model.predict(new_movie)

print("\nPredicted Movie Rating:",
      round(predicted_rating[0], 2))

<>:31: SyntaxWarning: invalid escape sequence '\d'
<>:35: SyntaxWarning: invalid escape sequence '\d'
<>:31: SyntaxWarning: invalid escape sequence '\d'
<>:35: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_1470/3917554635.py:31: SyntaxWarning: invalid escape sequence '\d'
  df['Year'] = df['Year'].astype(str).str.extract('(\d{4})')
/tmp/ipykernel_1470/3917554635.py:35: SyntaxWarning: invalid escape sequence '\d'
  df['Duration'] = df['Duration'].astype(str).str.extract('(\d+)')


Dataset Shape: (15509, 10)
                                 Name    Year Duration            Genre  \
0                                         NaN      NaN            Drama   
1  #Gadhvi (He thought he was Gandhi) -2019.0  109 min            Drama   
2                         #Homecoming -2021.0   90 min   Drama, Musical   
3                             #Yaaram -2019.0  110 min  Comedy, Romance   
4                   ...And Once Again -2010.0  105 min            Drama   

   Rating Votes            Director       Actor 1             Actor 2  \
0     NaN   NaN       J.S. Randhawa      Manmauji              Birbal   
1     7.0     8       Gaurav Bakshi  Rasika Dugal      Vivek Ghamande   
2     NaN   NaN  Soumyajit Majumdar  Sayani Gupta   Plabita Borthakur   
3     4.4    35          Ovais Khan       Prateik          Ishita Raj   
4     NaN   NaN        Amol Palekar  Rajat Kapoor  Rituparna Sengupta   

           Actor 3  
0  Rajendra Bhatia  
1    Arvind Jangid  
2       Roy Angana  